In [2]:
# ============================================================
# Cell 1 - Library dan konfigurasi utama
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

warnings.filterwarnings("ignore")


# ============================================================
# INPUT CONFIG
# ============================================================

DBSCAN_INPUT_ROOT = Path("/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset")


# ============================================================
# OUTPUT CONFIG
# ============================================================

SAMPLED_OUT_ROOT = Path("/media/spell/Spell-lab/Lidar/E.Sampled Dataset")


# ============================================================
# MAIN SAMPLING CONFIG
# ============================================================
# Ubah nilai ini saja untuk membuat output N berbeda.
# Contoh: 256, 369, 384, 512, 1024

N_TARGET = 512

N_FOLDER_NAME = f"N{N_TARGET:04d}"

SAMPLED_RUN_ROOT = SAMPLED_OUT_ROOT / N_FOLDER_NAME
DEV_OUT_ROOT = SAMPLED_RUN_ROOT / "Dataset Development"
TEST_OUT_ROOT = SAMPLED_RUN_ROOT / "Dataset Testing"
SUMMARY_DIR = SAMPLED_RUN_ROOT / "Summary"
LOG_DIR = SAMPLED_RUN_ROOT / "Logs"


# ============================================================
# SAMPLING BEHAVIOR
# ============================================================

OVERWRITE_OUTPUT = True
FLOAT_FORMAT = "%.6f"

SAMPLING_COORD_COLUMNS = ["X_corr", "Y_corr", "Z_level"]

REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

OUTPUT_COLUMNS = REQUIRED_COLUMNS.copy()


# ============================================================
# DATASET CONFIG
# ============================================================

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia", "Afi", "Aswangga", "Bustan",
    "Dilia", "Eldivo", "Fathir", "Lina",
    "Manda", "Miftah", "Teguh", "Tsamara",
]

TEST_ROOMS = ["Controlled Room", "Uncontrolled Room"]

TEST_SUBJECTS = [
    "Kanaya", "Naila", "Nana", "Rega", "Zaira",
]

FILE_IDS = list(range(1, 10))


print("===== FIXED-N SAMPLING CONFIG =====")
print(f"Input root       : {DBSCAN_INPUT_ROOT}")
print(f"Output root      : {SAMPLED_RUN_ROOT}")
print(f"N_TARGET         : {N_TARGET}")
print(f"Sampling coords  : {SAMPLING_COORD_COLUMNS}")
print(f"Overwrite output : {OVERWRITE_OUTPUT}")

===== FIXED-N SAMPLING CONFIG =====
Input root       : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset
Output root      : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512
N_TARGET         : 512
Sampling coords  : ['X_corr', 'Y_corr', 'Z_level']
Overwrite output : True


In [3]:
# ============================================================
# Cell 2 - Buat folder output dan simpan konfigurasi
# ============================================================

for d in [SAMPLED_OUT_ROOT, SAMPLED_RUN_ROOT, DEV_OUT_ROOT, TEST_OUT_ROOT, SUMMARY_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

config = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "input_root": str(DBSCAN_INPUT_ROOT),
    "sampled_out_root": str(SAMPLED_OUT_ROOT),
    "sampled_run_root": str(SAMPLED_RUN_ROOT),
    "n_target": int(N_TARGET),
    "n_folder_name": N_FOLDER_NAME,
    "sampling_strategy": {
        "M_greater_than_N": "centroid_initialized_farthest_point_sampling",
        "M_less_than_N": "retain_all_original_points_plus_uniform_bootstrap_duplicates",
        "M_equal_N": "unchanged",
    },
    "sampling_coord_columns": SAMPLING_COORD_COLUMNS,
    "required_columns": REQUIRED_COLUMNS,
    "output_columns": OUTPUT_COLUMNS,
    "overwrite_output": OVERWRITE_OUTPUT,
    "float_format": FLOAT_FORMAT,
}

config_path = SUMMARY_DIR / "fixedN_sampling_config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")

Config saved to: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/fixedN_sampling_config.json


In [4]:
# ============================================================
# Cell 3 - Build manifest input-output
# ============================================================

def build_sampling_manifest():
    records = []

    # Dataset Development
    for activity in ACTIVITIES:
        for subject in DEV_SUBJECTS:
            for file_id in FILE_IDS:
                file_name = f"{file_id}.csv"

                input_path = (
                    DBSCAN_INPUT_ROOT
                    / "Dataset Development"
                    / activity
                    / subject
                    / file_name
                )

                output_path = (
                    DEV_OUT_ROOT
                    / activity
                    / subject
                    / file_name
                )

                summary_path = (
                    SUMMARY_DIR
                    / "per_file_frame_summary"
                    / "Dataset Development"
                    / activity
                    / subject
                    / f"{file_id}_sampling_summary.csv"
                )

                records.append({
                    "dataset": "development",
                    "room": "",
                    "activity": activity,
                    "subject": subject,
                    "file_id": file_id,
                    "file_name": file_name,
                    "input_path": input_path,
                    "output_path": output_path,
                    "summary_path": summary_path,
                    "exists": input_path.exists(),
                })

    # Dataset Testing
    for room in TEST_ROOMS:
        for activity in ACTIVITIES:
            for subject in TEST_SUBJECTS:
                for file_id in FILE_IDS:
                    file_name = f"{file_id}.csv"

                    input_path = (
                        DBSCAN_INPUT_ROOT
                        / "Dataset Testing"
                        / room
                        / activity
                        / subject
                        / file_name
                    )

                    output_path = (
                        TEST_OUT_ROOT
                        / room
                        / activity
                        / subject
                        / file_name
                    )

                    summary_path = (
                        SUMMARY_DIR
                        / "per_file_frame_summary"
                        / "Dataset Testing"
                        / room
                        / activity
                        / subject
                        / f"{file_id}_sampling_summary.csv"
                    )

                    records.append({
                        "dataset": "testing",
                        "room": room,
                        "activity": activity,
                        "subject": subject,
                        "file_id": file_id,
                        "file_name": file_name,
                        "input_path": input_path,
                        "output_path": output_path,
                        "summary_path": summary_path,
                        "exists": input_path.exists(),
                    })

    return pd.DataFrame(records)


manifest_df = build_sampling_manifest()

existing_df = manifest_df[manifest_df["exists"]].copy()
missing_df = manifest_df[~manifest_df["exists"]].copy()

manifest_save = manifest_df.copy()

for col in ["input_path", "output_path", "summary_path"]:
    manifest_save[col] = manifest_save[col].astype(str)

manifest_path = SUMMARY_DIR / "sampling_manifest.csv"
missing_path = SUMMARY_DIR / "missing_input_files.csv"

manifest_save.to_csv(manifest_path, index=False)

missing_save = missing_df.copy()
for col in ["input_path", "output_path", "summary_path"]:
    missing_save[col] = missing_save[col].astype(str)

missing_save.to_csv(missing_path, index=False)

print("===== MANIFEST SUMMARY =====")
print(f"Total expected files : {len(manifest_df)}")
print(f"Existing files       : {len(existing_df)}")
print(f"Missing files        : {len(missing_df)}")
print(f"Manifest saved       : {manifest_path}")
print(f"Missing saved        : {missing_path}")

if len(existing_df) == 0:
    raise FileNotFoundError(
        "Tidak ada file input DBSCAN ditemukan. "
        "Cek DBSCAN_INPUT_ROOT dan struktur folder hasil DBSCAN."
    )

===== MANIFEST SUMMARY =====
Total expected files : 792
Existing files       : 792
Missing files        : 0
Manifest saved       : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/sampling_manifest.csv
Missing saved        : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/missing_input_files.csv


In [5]:
# ============================================================
# Cell 4 - Helper seed deterministic
# ============================================================

def stable_int_seed(*items, max_value=2**32 - 1):
    """
    Membuat seed deterministic dari metadata frame.
    Tujuannya agar hasil sampling reproducible walaupun notebook dijalankan ulang.
    """
    key = "|".join(str(x) for x in items)
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return int(digest[:16], 16) % max_value

In [6]:
# ============================================================
# Cell 5 - Helper validasi kolom dan numeric cleaning ringan
# ============================================================

def validate_required_columns(df, required_columns):
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        return False, missing_cols
    return True, []


def prepare_input_dataframe(df):
    """
    Menjaga hanya kolom yang dibutuhkan dan memastikan kolom numerik penting valid.
    Tidak melakukan filtering geometris lagi karena input sudah hasil DBSCAN main cluster.
    """
    df = df[REQUIRED_COLUMNS].copy()

    numeric_cols = [
        "frame_id",
        "Timestamp",
        "X", "Y", "Z",
        "X_corr", "Y_corr", "Z_corr", "Z_level",
        "Reflectivity",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop baris yang tidak bisa dipakai untuk sampling.
    df = df.dropna(subset=["frame_id"] + SAMPLING_COORD_COLUMNS).copy()

    df["frame_id"] = df["frame_id"].astype(int)

    # Drop non-finite pada koordinat sampling.
    finite_mask = np.ones(len(df), dtype=bool)
    for col in SAMPLING_COORD_COLUMNS:
        finite_mask &= np.isfinite(df[col].to_numpy())

    df = df.loc[finite_mask].copy()

    return df

In [7]:
# ============================================================
# Cell 6 - Helper centroid-initialized Farthest Point Sampling
# ============================================================

def farthest_point_sampling_indices(points, n_target):
    """
    points: numpy array shape [M, 3]
    n_target: jumlah titik yang dipilih

    Strategi:
    - titik awal = titik paling dekat centroid
    - iteratif pilih titik dengan jarak minimum terbesar terhadap titik terpilih
    """

    points = np.asarray(points, dtype=np.float64)
    m = points.shape[0]

    if n_target > m:
        raise ValueError("FPS hanya untuk kasus n_target <= jumlah point.")
    
    if n_target == m:
        return np.arange(m)

    centroid = points.mean(axis=0)
    dist_to_centroid = np.sum((points - centroid) ** 2, axis=1)
    first_idx = int(np.argmin(dist_to_centroid))

    selected = np.empty(n_target, dtype=np.int64)
    selected[0] = first_idx

    min_dist = np.sum((points - points[first_idx]) ** 2, axis=1)

    for i in range(1, n_target):
        next_idx = int(np.argmax(min_dist))
        selected[i] = next_idx

        dist_to_new = np.sum((points - points[next_idx]) ** 2, axis=1)
        min_dist = np.minimum(min_dist, dist_to_new)

    return selected

In [8]:
# ============================================================
# Cell 7 - Helper sampling per frame
# ============================================================

def sample_one_frame(frame_df, row_meta):
    """
    Input:
    - frame_df: data satu frame hasil DBSCAN main cluster
    - row_meta: metadata dataset/file untuk deterministic seed

    Output:
    - sampled_df: frame dengan jumlah row = N_TARGET
    - summary: audit sampling frame
    """

    frame_id = int(frame_df["frame_id"].iloc[0])
    m_points = len(frame_df)

    base_summary = {
        "dataset": row_meta["dataset"],
        "room": row_meta["room"],
        "activity": row_meta["activity"],
        "subject": row_meta["subject"],
        "file_id": int(row_meta["file_id"]),
        "file_name": row_meta["file_name"],
        "frame_id": frame_id,
        "n_original_points": int(m_points),
        "n_target": int(N_TARGET),
    }

    if m_points == 0:
        summary = {
            **base_summary,
            "sampling_status": "invalid_empty_frame",
            "sampling_mode": "invalid",
            "n_sampled_points": 0,
            "n_added": 0,
            "n_removed": 0,
            "duplicate_ratio": np.nan,
        }
        return pd.DataFrame(columns=OUTPUT_COLUMNS), summary

    frame_df = frame_df[OUTPUT_COLUMNS].copy().reset_index(drop=True)

    points = frame_df[SAMPLING_COORD_COLUMNS].to_numpy(dtype=np.float64)

    # Case 1: M > N, downsample pakai FPS
    if m_points > N_TARGET:
        selected_idx = farthest_point_sampling_indices(points, N_TARGET)
        sampled_df = frame_df.iloc[selected_idx].copy().reset_index(drop=True)

        summary = {
            **base_summary,
            "sampling_status": "valid",
            "sampling_mode": "downsample_fps",
            "n_sampled_points": len(sampled_df),
            "n_added": 0,
            "n_removed": int(m_points - N_TARGET),
            "duplicate_ratio": 0.0,
        }

        return sampled_df, summary

    # Case 2: M < N, retain all + uniform bootstrap
    if m_points < N_TARGET:
        n_add = N_TARGET - m_points

        seed = stable_int_seed(
            row_meta["dataset"],
            row_meta["room"],
            row_meta["activity"],
            row_meta["subject"],
            row_meta["file_id"],
            frame_id,
            N_TARGET,
        )

        rng = np.random.default_rng(seed)

        duplicate_idx = rng.choice(
            np.arange(m_points),
            size=n_add,
            replace=True,
        )

        duplicate_df = frame_df.iloc[duplicate_idx].copy()

        sampled_df = pd.concat(
            [frame_df, duplicate_df],
            ignore_index=True,
        )

        summary = {
            **base_summary,
            "sampling_status": "valid",
            "sampling_mode": "upsample_bootstrap_retain_all",
            "n_sampled_points": len(sampled_df),
            "n_added": int(n_add),
            "n_removed": 0,
            "duplicate_ratio": float(n_add / N_TARGET),
        }

        return sampled_df, summary

    # Case 3: M == N, unchanged
    sampled_df = frame_df.copy().reset_index(drop=True)

    summary = {
        **base_summary,
        "sampling_status": "valid",
        "sampling_mode": "unchanged",
        "n_sampled_points": len(sampled_df),
        "n_added": 0,
        "n_removed": 0,
        "duplicate_ratio": 0.0,
    }

    return sampled_df, summary

In [9]:
# ============================================================
# Cell 8 - Helper proses satu file
# ============================================================

def process_one_file_sampling(row):
    input_path = Path(row["input_path"])
    output_path = Path(row["output_path"])
    summary_path = Path(row["summary_path"])

    file_info = {
        "dataset": row["dataset"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_id": int(row["file_id"]),
        "file_name": row["file_name"],
        "input_path": str(input_path),
        "output_path": str(output_path),
        "summary_path": str(summary_path),
        "n_target": int(N_TARGET),
    }

    if not input_path.exists():
        return {
            **file_info,
            "file_status": "missing_input",
            "n_input_rows": 0,
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
        }

    if output_path.exists() and not OVERWRITE_OUTPUT:
        return {
            **file_info,
            "file_status": "skipped_output_exists",
            "n_input_rows": np.nan,
            "n_output_rows": np.nan,
            "n_frames_total": np.nan,
            "n_frames_valid": np.nan,
        }

    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        return {
            **file_info,
            "file_status": f"read_error: {e}",
            "n_input_rows": 0,
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
        }

    valid_cols, missing_cols = validate_required_columns(df, REQUIRED_COLUMNS)
    if not valid_cols:
        return {
            **file_info,
            "file_status": f"missing_columns: {missing_cols}",
            "n_input_rows": len(df),
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
        }

    df = prepare_input_dataframe(df)

    if len(df) == 0:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        summary_path.parent.mkdir(parents=True, exist_ok=True)

        pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(output_path, index=False)
        pd.DataFrame().to_csv(summary_path, index=False)

        return {
            **file_info,
            "file_status": "empty_after_prepare",
            "n_input_rows": 0,
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
        }

    sampled_frames = []
    frame_summaries = []

    row_meta = {
        "dataset": row["dataset"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_id": int(row["file_id"]),
        "file_name": row["file_name"],
    }

    for frame_id, frame_df in df.groupby("frame_id", sort=True):
        sampled_df, summary = sample_one_frame(frame_df, row_meta)
        frame_summaries.append(summary)

        if len(sampled_df) > 0:
            sampled_frames.append(sampled_df)

    if sampled_frames:
        output_df = pd.concat(sampled_frames, ignore_index=True)
    else:
        output_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

    frame_summary_df = pd.DataFrame(frame_summaries)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.parent.mkdir(parents=True, exist_ok=True)

    output_df.to_csv(output_path, index=False, float_format=FLOAT_FORMAT)
    frame_summary_df.to_csv(summary_path, index=False, float_format=FLOAT_FORMAT)

    n_frames_total = len(frame_summary_df)
    n_frames_valid = int((frame_summary_df["sampling_status"] == "valid").sum()) if n_frames_total > 0 else 0

    mode_counts = (
        frame_summary_df["sampling_mode"].value_counts().to_dict()
        if n_frames_total > 0 and "sampling_mode" in frame_summary_df.columns
        else {}
    )

    return {
        **file_info,
        "file_status": "processed",
        "n_input_rows": len(df),
        "n_output_rows": len(output_df),
        "n_frames_total": n_frames_total,
        "n_frames_valid": n_frames_valid,
        "expected_output_rows": int(n_frames_valid * N_TARGET),
        "output_row_check_passed": bool(len(output_df) == n_frames_valid * N_TARGET),
        "n_downsample_fps": int(mode_counts.get("downsample_fps", 0)),
        "n_upsample_bootstrap": int(mode_counts.get("upsample_bootstrap_retain_all", 0)),
        "n_unchanged": int(mode_counts.get("unchanged", 0)),
        "mean_original_points": (
            frame_summary_df["n_original_points"].mean()
            if n_frames_total > 0 else np.nan
        ),
        "mean_duplicate_ratio": (
            frame_summary_df["duplicate_ratio"].mean()
            if n_frames_total > 0 and "duplicate_ratio" in frame_summary_df.columns else np.nan
        ),
    }

In [10]:
# ============================================================
# Cell 9 - Mini test satu file
# ============================================================

test_row = existing_df.iloc[0].to_dict()

print("===== MINI TEST SAMPLING =====")
print(f"Input : {test_row['input_path']}")
print(f"Output: {test_row['output_path']}")
print(f"N     : {N_TARGET}")

mini_result = process_one_file_sampling(test_row)

print("\n===== MINI TEST RESULT =====")
for k, v in mini_result.items():
    print(f"{k}: {v}")

mini_output_path = Path(mini_result["output_path"])
mini_summary_path = Path(mini_result["summary_path"])

if mini_output_path.exists():
    mini_df = pd.read_csv(mini_output_path)
    print("\nMini output preview:")
    display(mini_df.head())
    print(f"Mini output shape: {mini_df.shape}")

    if len(mini_df) > 0:
        check_counts = mini_df.groupby("frame_id").size()
        print("\nFrame point count check:")
        display(check_counts.describe())
        print("Unique point counts per frame:", sorted(check_counts.unique().tolist())[:10])

if mini_summary_path.exists():
    mini_summary_df = pd.read_csv(mini_summary_path)
    print("\nMini frame summary preview:")
    display(mini_summary_df.head())
    print(f"Mini summary shape: {mini_summary_df.shape}")
    display(mini_summary_df["sampling_mode"].value_counts().to_frame("count"))

===== MINI TEST SAMPLING =====
Input : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Dataset Development/Bungkuk/Adelia/1.csv
Output: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Dataset Development/Bungkuk/Adelia/1.csv
N     : 512

===== MINI TEST RESULT =====
dataset: development
room: 
activity: Bungkuk
subject: Adelia
file_id: 1
file_name: 1.csv
input_path: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Dataset Development/Bungkuk/Adelia/1.csv
output_path: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Dataset Development/Bungkuk/Adelia/1.csv
summary_path: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/per_file_frame_summary/Dataset Development/Bungkuk/Adelia/1_sampling_summary.csv
n_target: 512
file_status: processed
n_input_rows: 46065
n_output_rows: 28160
n_frames_total: 55
n_frames_valid: 55
expected_output_rows: 28160
output_row_check_passed: True
n_downsample_fps: 55
n_upsample_bootstrap: 0
n_unchanged: 0
mean_original_points: 837.5454545454545
mean

,frame_id,Timestamp,X,Y,Z,X_corr,Y_corr,Z_corr,Z_level,Reflectivity
0,0,4619275692940,0.796,0.655,0.394,0.887933,0.655,0.020681,0.907353,19.0
1,0,4619302592940,1.471,0.526,-0.151,1.269363,0.526,-0.758524,0.128147,151.0
2,0,4619270412940,0.370,1.128,0.683,0.623982,1.128,0.462639,1.349311,2.0
3,0,4619258892940,0.763,0.665,1.014,1.120048,0.665,0.596538,1.483210,5.0
4,0,4619313652940,1.047,0.875,-0.057,0.924815,0.875,-0.494141,0.392531,13.0


Mini output shape: (28160, 10)

Frame point count check:


count     55.0
mean     512.0
std        0.0
min      512.0
25%      512.0
50%      512.0
75%      512.0
max      512.0
dtype: float64

Unique point counts per frame: [512]

Mini frame summary preview:


,dataset,room,activity,subject,file_id,file_name,frame_id,n_original_points,n_target,sampling_status,sampling_mode,n_sampled_points,n_added,n_removed,duplicate_ratio
0,development,NaN,Bungkuk,Adelia,1,1.csv,0,1056,512,valid,downsample_fps,512,0,544,0.0
1,development,NaN,Bungkuk,Adelia,1,1.csv,1,955,512,valid,downsample_fps,512,0,443,0.0
2,development,NaN,Bungkuk,Adelia,1,1.csv,2,1024,512,valid,downsample_fps,512,0,512,0.0
3,development,NaN,Bungkuk,Adelia,1,1.csv,3,998,512,valid,downsample_fps,512,0,486,0.0
4,development,NaN,Bungkuk,Adelia,1,1.csv,4,1033,512,valid,downsample_fps,512,0,521,0.0


Mini summary shape: (55, 15)


,count
sampling_mode,
downsample_fps,55


In [11]:
# ============================================================
# Cell 10 - Apply sampling ke seluruh dataset
# ============================================================

file_results = []

for _, row in tqdm(existing_df.iterrows(), total=len(existing_df), desc=f"Fixed-N Sampling {N_FOLDER_NAME}"):
    result = process_one_file_sampling(row)
    file_results.append(result)

file_summary_df = pd.DataFrame(file_results)

file_summary_path = SUMMARY_DIR / "sampling_file_summary.csv"
file_summary_df.to_csv(file_summary_path, index=False, float_format=FLOAT_FORMAT)

print("===== FULL SAMPLING DONE =====")
print(f"N target          : {N_TARGET}")
print(f"Output root       : {SAMPLED_RUN_ROOT}")
print(f"Total files       : {len(file_summary_df)}")
print(f"File summary saved: {file_summary_path}")

display(file_summary_df["file_status"].value_counts(dropna=False).to_frame("count"))

Fixed-N Sampling N0512:   0%|          | 0/792 [00:00<?, ?it/s]

===== FULL SAMPLING DONE =====
N target          : 512
Output root       : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512
Total files       : 792
File summary saved: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/sampling_file_summary.csv


,count
file_status,
processed,792


In [12]:
# ============================================================
# Cell 11 - Merge semua frame summary
# ============================================================

frame_summary_paths = list((SUMMARY_DIR / "per_file_frame_summary").rglob("*_sampling_summary.csv"))

all_frame_summaries = []

for path in tqdm(frame_summary_paths, desc="Merging sampling summaries"):
    try:
        temp = pd.read_csv(path)
        temp["summary_file"] = str(path)
        all_frame_summaries.append(temp)
    except Exception as e:
        print(f"Failed reading {path}: {e}")

if all_frame_summaries:
    global_sampling_summary_df = pd.concat(all_frame_summaries, ignore_index=True)
else:
    global_sampling_summary_df = pd.DataFrame()

global_sampling_summary_path = SUMMARY_DIR / "sampling_frame_summary_global.csv"
global_sampling_summary_df.to_csv(global_sampling_summary_path, index=False, float_format=FLOAT_FORMAT)

print("===== GLOBAL SAMPLING SUMMARY =====")
print(f"Total frame summaries : {len(global_sampling_summary_df)}")
print(f"Saved to              : {global_sampling_summary_path}")

if len(global_sampling_summary_df) > 0:
    display(global_sampling_summary_df.head())
    display(global_sampling_summary_df["sampling_mode"].value_counts(dropna=False).to_frame("count"))

Merging sampling summaries:   0%|          | 0/792 [00:00<?, ?it/s]

===== GLOBAL SAMPLING SUMMARY =====
Total frame summaries : 47371
Saved to              : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/sampling_frame_summary_global.csv


,dataset,room,activity,subject,file_id,file_name,frame_id,n_original_points,n_target,sampling_status,sampling_mode,n_sampled_points,n_added,n_removed,duplicate_ratio,summary_file
0,testing,Uncontrolled Room,Jongkok,Kanaya,2,2.csv,0,1267,512,valid,downsample_fps,512,0,755,0.0,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...
1,testing,Uncontrolled Room,Jongkok,Kanaya,2,2.csv,1,1151,512,valid,downsample_fps,512,0,639,0.0,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...
2,testing,Uncontrolled Room,Jongkok,Kanaya,2,2.csv,2,1150,512,valid,downsample_fps,512,0,638,0.0,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...
3,testing,Uncontrolled Room,Jongkok,Kanaya,2,2.csv,3,1124,512,valid,downsample_fps,512,0,612,0.0,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...
4,testing,Uncontrolled Room,Jongkok,Kanaya,2,2.csv,4,1159,512,valid,downsample_fps,512,0,647,0.0,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...


,count
sampling_mode,
upsample_bootstrap_retain_all,33217
downsample_fps,14099
unchanged,55


In [13]:
# ============================================================
# Cell 12 - Validasi jumlah point per frame hasil sampling
# ============================================================

if len(global_sampling_summary_df) == 0:
    raise ValueError("Global sampling summary kosong. Jalankan Cell 11 terlebih dahulu.")

valid_summary_df = global_sampling_summary_df[
    global_sampling_summary_df["sampling_status"] == "valid"
].copy()

point_check = valid_summary_df["n_sampled_points"].value_counts().sort_index()

print("===== FIXED-N CHECK FROM SUMMARY =====")
display(point_check.to_frame("frame_count"))

all_fixed_n = (
    len(point_check) == 1
    and int(point_check.index[0]) == int(N_TARGET)
)

print(f"All valid frames have N_TARGET={N_TARGET}: {all_fixed_n}")

if not all_fixed_n:
    bad_frames = valid_summary_df[valid_summary_df["n_sampled_points"] != N_TARGET]
    print("Bad frames:")
    display(bad_frames.head(20))

===== FIXED-N CHECK FROM SUMMARY =====


,frame_count
n_sampled_points,
512,47371


All valid frames have N_TARGET=512: True


In [14]:
# ============================================================
# Cell 13 - Statistik sampling global
# ============================================================

stats = {
    "n_target": int(N_TARGET),
    "n_valid_frames": len(valid_summary_df),
    "n_downsample_fps": int((valid_summary_df["sampling_mode"] == "downsample_fps").sum()),
    "n_upsample_bootstrap": int((valid_summary_df["sampling_mode"] == "upsample_bootstrap_retain_all").sum()),
    "n_unchanged": int((valid_summary_df["sampling_mode"] == "unchanged").sum()),
    "downsample_ratio": float((valid_summary_df["sampling_mode"] == "downsample_fps").mean()),
    "upsample_ratio": float((valid_summary_df["sampling_mode"] == "upsample_bootstrap_retain_all").mean()),
    "unchanged_ratio": float((valid_summary_df["sampling_mode"] == "unchanged").mean()),
    "original_points_min": valid_summary_df["n_original_points"].min(),
    "original_points_mean": valid_summary_df["n_original_points"].mean(),
    "original_points_median": valid_summary_df["n_original_points"].median(),
    "original_points_max": valid_summary_df["n_original_points"].max(),
    "duplicate_ratio_mean": valid_summary_df["duplicate_ratio"].mean(),
    "duplicate_ratio_median": valid_summary_df["duplicate_ratio"].median(),
    "duplicate_ratio_max": valid_summary_df["duplicate_ratio"].max(),
}

sampling_global_stats_df = pd.DataFrame([stats])

sampling_global_stats_path = SUMMARY_DIR / "sampling_global_stats.csv"
sampling_global_stats_df.to_csv(sampling_global_stats_path, index=False, float_format=FLOAT_FORMAT)

print("===== SAMPLING GLOBAL STATS =====")
display(sampling_global_stats_df)
print(f"Saved to: {sampling_global_stats_path}")

===== SAMPLING GLOBAL STATS =====


,n_target,n_valid_frames,n_downsample_fps,n_upsample_bootstrap,n_unchanged,downsample_ratio,upsample_ratio,unchanged_ratio,original_points_min,original_points_mean,original_points_median,original_points_max,duplicate_ratio_mean,duplicate_ratio_median,duplicate_ratio_max
0,512,47371,14099,33217,55,0.297629,0.70121,0.001161,40,433.819869,369.0,1666,0.302213,0.279297,0.921875


Saved to: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/sampling_global_stats.csv


In [15]:
# ============================================================
# Cell 14 - Statistik sampling per dataset, room, activity, subject
# ============================================================

group_cols = ["dataset", "room", "activity", "subject"]

group_sampling_stats_df = (
    valid_summary_df
    .groupby(group_cols, dropna=False)
    .agg(
        n_valid_frames=("frame_id", "count"),
        mean_original_points=("n_original_points", "mean"),
        median_original_points=("n_original_points", "median"),
        min_original_points=("n_original_points", "min"),
        max_original_points=("n_original_points", "max"),
        mean_duplicate_ratio=("duplicate_ratio", "mean"),
        max_duplicate_ratio=("duplicate_ratio", "max"),
        n_downsample_fps=("sampling_mode", lambda x: (x == "downsample_fps").sum()),
        n_upsample_bootstrap=("sampling_mode", lambda x: (x == "upsample_bootstrap_retain_all").sum()),
        n_unchanged=("sampling_mode", lambda x: (x == "unchanged").sum()),
    )
    .reset_index()
)

group_sampling_stats_df["downsample_ratio"] = (
    group_sampling_stats_df["n_downsample_fps"] / group_sampling_stats_df["n_valid_frames"]
)

group_sampling_stats_df["upsample_ratio"] = (
    group_sampling_stats_df["n_upsample_bootstrap"] / group_sampling_stats_df["n_valid_frames"]
)

group_sampling_stats_df["unchanged_ratio"] = (
    group_sampling_stats_df["n_unchanged"] / group_sampling_stats_df["n_valid_frames"]
)

group_sampling_stats_path = SUMMARY_DIR / "sampling_group_stats.csv"
group_sampling_stats_df.to_csv(group_sampling_stats_path, index=False, float_format=FLOAT_FORMAT)

print("===== SAMPLING GROUP STATS =====")
display(group_sampling_stats_df.head(20))
print(f"Saved to: {group_sampling_stats_path}")

===== SAMPLING GROUP STATS =====


,dataset,room,activity,subject,n_valid_frames,mean_original_points,median_original_points,min_original_points,max_original_points,mean_duplicate_ratio,max_duplicate_ratio,n_downsample_fps,n_upsample_bootstrap,n_unchanged,downsample_ratio,upsample_ratio,unchanged_ratio
0,development,NaN,Bungkuk,Adelia,518,496.536680,482.0,91,1385,0.239005,0.822266,225,292,1,0.434363,0.563707,0.001931
1,development,NaN,Bungkuk,Afi,523,448.330784,416.0,66,1155,0.279073,0.871094,210,313,0,0.401530,0.598470,0.000000
2,development,NaN,Bungkuk,Aswangga,517,387.735010,319.0,56,1269,0.359685,0.890625,153,363,1,0.295938,0.702128,0.001934
3,development,NaN,Bungkuk,Bustan,498,592.461847,463.5,79,1666,0.236865,0.845703,239,259,0,0.479920,0.520080,0.000000
4,development,NaN,Bungkuk,Dilia,596,567.929530,488.5,84,1385,0.198980,0.835938,275,319,2,0.461409,0.535235,0.003356
5,development,NaN,Bungkuk,Eldivo,553,427.929476,376.0,63,1466,0.318391,0.876953,183,368,2,0.330922,0.665461,0.003617
6,development,NaN,Bungkuk,Fathir,510,425.029412,391.0,71,1197,0.294221,0.861328,188,322,0,0.368627,0.631373,0.000000
7,development,NaN,Bungkuk,Lina,556,489.645683,426.0,101,1322,0.256190,0.802734,214,342,0,0.384892,0.615108,0.000000
8,development,NaN,Bungkuk,Manda,563,340.797513,265.0,40,1097,0.399780,0.921875,127,433,3,0.225577,0.769094,0.005329
9,development,NaN,Bungkuk,Miftah,521,395.719770,347.0,77,1304,0.338892,0.849609,159,362,0,0.305182,0.694818,0.000000


Saved to: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512/Summary/sampling_group_stats.csv


In [16]:
# ============================================================
# Cell 15 - Final sanity check
# ============================================================

print("===== FINAL SANITY CHECK =====")

print("\nInput root:")
print(DBSCAN_INPUT_ROOT)

print("\nOutput root:")
print(SAMPLED_RUN_ROOT)

print("\nFolder exists:")
print("Dataset Development:", DEV_OUT_ROOT.exists())
print("Dataset Testing    :", TEST_OUT_ROOT.exists())
print("Summary            :", SUMMARY_DIR.exists())

print("\nFile status:")
display(file_summary_df["file_status"].value_counts(dropna=False).to_frame("count"))

print("\nSampling mode:")
display(global_sampling_summary_df["sampling_mode"].value_counts(dropna=False).to_frame("count"))

print("\nFixed-N summary check:")
display(point_check.to_frame("frame_count"))

print("\nGlobal stats:")
display(sampling_global_stats_df)

===== FINAL SANITY CHECK =====

Input root:
/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset

Output root:
/media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0512

Folder exists:
Dataset Development: True
Dataset Testing    : True
Summary            : True

File status:


,count
file_status,
processed,792



Sampling mode:


,count
sampling_mode,
upsample_bootstrap_retain_all,33217
downsample_fps,14099
unchanged,55



Fixed-N summary check:


,frame_count
n_sampled_points,
512,47371



Global stats:


,n_target,n_valid_frames,n_downsample_fps,n_upsample_bootstrap,n_unchanged,downsample_ratio,upsample_ratio,unchanged_ratio,original_points_min,original_points_mean,original_points_median,original_points_max,duplicate_ratio_mean,duplicate_ratio_median,duplicate_ratio_max
0,512,47371,14099,33217,55,0.297629,0.70121,0.001161,40,433.819869,369.0,1666,0.302213,0.279297,0.921875
